# Arabidopsis meristem (Jackson et al. 2019): BOSPERRUS buffer

Data: `results/physically_realized_graphs/jackson2019/` (`src/physically_realized_graphs/jackson2019_analysis.py`).
Degree and log10 betweenness are computed on the full segmented networks; the distance is the lateral (x-y)
distance of each centroid to the field-of-view edge. The BOSPERRUS buffer of a meristem keeps the cells beyond
the piecewise-linear knot of a score vs. that distance, fitted on its L1-L3 cells. Correlations: the four T0
meristems, L1-L3, as in the paper's Fig. 3D.

In [ ]:
import json
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.spatial import ConvexHull

from bosperrus.fit import PiecewiseLinearFit
from bosperrus.plotting import plot_fit

ROOT = Path("../..")
RES = ROOT / "results" / "physically_realized_graphs" / "jackson2019"
TEXTWIDTH = (8.5 * 2.54 - 4.0) / 2.54  # manuscript text width (letterpaper, 2 cm margins), inches
PL_BLUE = json.load(open(ROOT / "fit_palette.json"))["Piecewise Linear Fit"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "svg.fonttype": "none"})

cells = pd.read_csv(RES / "cells.csv")
t0 = cells[(cells.time == "T0") & cells.layer.between(1, 3)]

In [ ]:
def r2_panels(scenarios, props, figsize, wrap=None):
    """Pearson R² of the network property (last entry of `props`) with each cell-size property, one row per scenario panel."""
    *size, score = props
    r2 = {name: df[list(props)].astype(float).corr(method="pearson").loc[[score], size] ** 2 for name, df in scenarios.items()}
    fig, axes = plt.subplots(1, len(r2), figsize=figsize, constrained_layout=True)
    for ax, (name, m) in zip(axes, r2.items()):
        im = ax.imshow(m.values, cmap="magma_r", vmin=0, vmax=1)
        for j, v in enumerate(m.values[0]):
            ax.text(j, 0, f"{v:.2g}", ha="center", va="center", fontsize=7, color="white" if v > 0.55 else "black")
        ax.set_xticks(range(len(size)), [props[s] for s in size], fontsize=7)
        ax.set_yticks([0], [props[score]], fontsize=7)
        title = textwrap.fill(name, wrap) + "\n" if wrap else f"{name}\n"
        ax.set_title(title + f"n = {len(scenarios[name])}", fontsize=7)
        ax.spines[:].set_visible(False)
        ax.tick_params(length=0)
    cb = fig.colorbar(im, ax=axes, shrink=0.9, aspect=8)
    cb.set_label("Pearson R²", fontsize=7)
    cb.ax.tick_params(labelsize=6.5)
    return fig, r2

## Degree vs. cell size: no buffer, the paper's buffer, the BOSPERRUS buffer (degree knot)

In [ ]:
SIZE_PROPS = {"area": "area", "volume": "volume", "degree": "degree"}
size_scenarios = {
    "all cells, raw": t0,
    "central cells, raw (paper)": t0[t0.central],
    "BOSPERRUS buffer (degree knot)": t0[t0.beyond_knot_degree],
}
fig, size_r2 = r2_panels(size_scenarios, SIZE_PROPS, (TEXTWIDTH, TEXTWIDTH * 0.17))
fig.savefig("meristem_degree_size_r2.svg", dpi=300)
pd.DataFrame({name: {"degree–area": m.loc["degree", "area"], "degree–volume": m.loc["degree", "volume"],
                     "cells": len(size_scenarios[name])} for name, m in size_r2.items()}).round(2)

## Betweenness vs. cell size: no buffer, the paper's buffer, all cells corrected, BOSPERRUS buffer (BC knot)

In the third scenario log10 betweenness is BOSPERRUS-corrected per meristem and layer against the distance to the
field-of-view edge; area and volume stay raw.

In [ ]:
BC_PROPS = {"area": "area", "volume": "volume", "log_bc": "log10 BC"}
bc_scenarios = {
    "all cells, raw": t0,
    "central cells, raw (paper)": t0[t0.central],
    "all cells, BOSPERRUS": t0.assign(log_bc=t0.log_bc_corr),
    "BOSPERRUS buffer (log10 BC knot)": t0[t0.beyond_knot_log_bc],
}
fig, bc_r2 = r2_panels(bc_scenarios, BC_PROPS, (TEXTWIDTH, TEXTWIDTH * 0.17), wrap=22)
fig.savefig("meristem_bc_size_r2.svg", dpi=300)
pd.DataFrame({name: {"log10 BC–area": m.loc["log_bc", "area"], "log10 BC–volume": m.loc["log_bc", "volume"],
                     "cells": len(bc_scenarios[name])} for name, m in bc_r2.items()}).round(2)

## One meristem, L1: degree and betweenness with the BOSPERRUS buffer

Rep 2 T1, L1 cells (centroids, top view). Blue (the piecewise-linear colour of the fit palette): cells closer to
the field-of-view edge than the knot of that score, excluded. Black line: convex hull of the central L1 cells the
paper evaluated. Right column: the piecewise-linear fits the knots come from (L1-L3 of this meristem; refitted
here, seconds; bands: 5-95 % and 25-75 % per distance bin, black: bin mean).

In [ ]:
full = cells[(cells.rep == "Rep 2") & (cells.time == "T1")].set_index("label")
fit_cells = full[full.layer.between(1, 3)]
t = full[np.minimum(full.layer, 4) == 1]
cen = t[t.central][["x", "y"]].values
hull = cen[ConvexHull(cen).vertices]
hull = np.vstack([hull, hull[:1]])

fig, axes = plt.subplots(2, 3, figsize=(TEXTWIDTH, TEXTWIDTH * 0.62), constrained_layout=True,
                         gridspec_kw={"width_ratios": [1, 1, 1.1]})
s = 16
for row, (col, measure, label) in enumerate([("degree", "degree", "degree"), ("log_bc", "log10 BC", "log10 betweenness")]):
    excluded = ~t[f"beyond_knot_{col}"]
    kw = dict(cmap="YlOrRd", vmin=t[col].quantile(0.02), vmax=t[col].quantile(0.98), s=s, lw=0, rasterized=True)
    sc = axes[row, 0].scatter(t.x, t.y, c=t[col], **kw)
    axes[row, 0].set_title(f"{'AD'[row]}  {label}", loc="left", fontsize=7.5)
    ax = axes[row, 1]
    ax.scatter(t.x[~excluded], t.y[~excluded], c=t[col][~excluded], **kw)
    ax.scatter(t.x[excluded], t.y[excluded], color=PL_BLUE, s=s, lw=0, rasterized=True)
    ax.plot(hull[:, 0], hull[:, 1], color="k", lw=0.8)
    ax.set_title(f"{'BE'[row]}  buffer ({excluded.sum()} of {len(t)} excluded)", loc="left", fontsize=7.5)
    for a in axes[row, :2]:
        a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([]); a.spines[:].set_visible(False)
    cb = fig.colorbar(sc, ax=axes[row, :2], shrink=0.8, pad=0.02)
    cb.set_label(label, fontsize=7); cb.ax.tick_params(labelsize=6.5)
    fit = PiecewiseLinearFit(fit_cells[col].reset_index(drop=True), fit_cells["d"].reset_index(drop=True))
    fit.fit()
    knot = fit.params["piecewise_linear_b"]
    ax = axes[row, 2]
    plot_fit(ax, fit_cells["d"].values, fit_cells[col].values, fit.predict, bins=25,
             line_kwargs=dict(color=PL_BLUE, linewidth=1.2, label="piecewise linear fit"))
    ax.axvspan(0, knot, color=PL_BLUE, alpha=0.12, lw=0)
    ax.axvline(knot, color=PL_BLUE, lw=0.8, ls="--")
    ax.text(knot, 0.97, f" knot {knot:.2g} µm", transform=ax.get_xaxis_transform(), va="top", fontsize=6, color=PL_BLUE)
    ax.set_xlabel("distance to image edge (µm)", fontsize=7); ax.set_ylabel(label, fontsize=7)
    ax.tick_params(labelsize=6.5)
    ax.set_title(f"{'CF'[row]}  {measure} vs. distance", loc="left", fontsize=7.5)
axes[0, 0].plot([t.x.min(), t.x.min() + 20], [t.y.min() - 4] * 2, color="k", lw=1.2)  # 20 µm scale bar
axes[0, 0].text(t.x.min() + 10, t.y.min() - 6, "20 µm", ha="center", va="top", fontsize=6)
fig.legend(handles=[Patch(color=PL_BLUE, label="BOSPERRUS buffer (closer than the knot), excluded"),
                    Line2D([], [], color="k", lw=0.8, label="evaluated in the paper")],
           loc="outside lower center", ncol=2, frameon=False, fontsize=6, handlelength=1.2)
fig.savefig("meristem_L1_maps.svg", dpi=300)

## Degree and log10 betweenness vs. cell size over the exclusion distance, per meristem

Each meristem (L1-L3 cells): Pearson R² of the score with area and volume when only cells farther than a given
distance from the field-of-view edge are kept. Blue dashed: this meristem's knot for that score (the BOSPERRUS
buffer); black dotted: inner edge of the paper's central region (5th percentile of the central cells' distance);
grey: cells kept (right axis).

In [ ]:
all_knots = pd.read_csv(RES / "knots.csv")
cutoffs = np.arange(0, 40.5, 0.5)


def r2_vs_exclusion(score, label, out):
    knots = all_knots[all_knots.measure == score].set_index(["rep", "time"])
    fig, axes = plt.subplots(2, 4, figsize=(TEXTWIDTH, TEXTWIDTH * 0.5), sharex=True, sharey=True, constrained_layout=True)
    rows = []
    for ax, ((rep, time), k) in zip(axes.T.flat, knots.iterrows()):
        t = cells[(cells.rep == rep) & (cells.time == time) & cells.layer.between(1, 3)]
        r2 = pd.DataFrame([{**(t[t.d > c][[score, "area", "volume"]].astype(float).corr().loc[score, ["area", "volume"]] ** 2),
                            "n": (t.d > c).sum()} for c in cutoffs], index=cutoffs)
        ax.plot(cutoffs, r2.area, color="k", lw=1, label=f"{label}–area")
        ax.plot(cutoffs, r2.volume, color="k", lw=1, ls="--", label=f"{label}–volume")
        ax.axvline(k.knot_um, color=PL_BLUE, lw=1, ls="--")
        ax.axvline(k.paper_edge_um, color="k", lw=0.8, ls=":")
        axn = ax.twinx()
        axn.plot(cutoffs, r2.n, color="0.6", lw=0.8)
        axn.set_ylim(0, 900)
        axn.tick_params(labelsize=6, colors="0.5")
        if ax not in axes[:, -1]:
            axn.set_yticklabels([])
        ax.set_title(f"{rep} {time}", fontsize=7)
        ax.tick_params(labelsize=6)
        at = lambda c: t[t.d > c][[score, "area", "volume"]].astype(float).corr().loc[score] ** 2
        rows.append(dict(meristem=f"{rep} {time}", **{f"{p} {where}": at(cut)[p] for where, cut in [("knot", k.knot_um), ("paper", k.paper_edge_um)] for p in ["area", "volume"]},
                         **{f"cells {where}": (t.d > cut).sum() for where, cut in [("knot", k.knot_um), ("paper", k.paper_edge_um)]}))
    axes[0, 0].set_ylim(0, 1)
    fig.supxlabel("exclusion distance from the field-of-view edge (µm)", fontsize=6.5)
    for ax in axes[:, 0]:
        ax.set_ylabel("Pearson R²", fontsize=6.5)
    axes[0, 0].legend(fontsize=5.5, frameon=False, loc="lower right")
    fig.savefig(out, dpi=300)
    plt.show()
    return pd.DataFrame(rows).set_index("meristem").round(2)


r2_vs_exclusion("degree", "degree", "meristem_r2_vs_exclusion.svg")

In [ ]:
r2_vs_exclusion("log_bc", "log10 BC", "meristem_bc_r2_vs_exclusion.svg")